In [1]:
import pandas as pd
import numpy as np
import re
import string

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [7]:
from google.colab import files
uploaded = files.upload()


Saving IMDB Dataset.csv to IMDB Dataset (1).csv


In [8]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
print("Total Samples:", df.shape[0])
print("\nClass Distribution:\n", df['sentiment'].value_counts())

df.sample(5)


Total Samples: 50000

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
16332,Some movies are not for everyone. This accurat...,negative
775,"This video is so hilariously funny, it makes e...",positive
31767,"Adapted from Sam Shepard's play, this movie re...",positive
29754,Like a lot of horror fans out there that went ...,positive
32707,"La Sanguisuga Conduce la Danza, or The Bloodsu...",negative


In [10]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', '', text)

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'\d+', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)


In [11]:
df['clean_text'] = df['review'].apply(preprocess_text)
df.head()


,review,sentiment,clean_text
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [12]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [14]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)


In [15]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [16]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [17]:
lr = LogisticRegression()
lr.fit(X_train_bow, y_train)

print("Logistic Regression (BoW)")
evaluate_model(lr, X_test_bow, y_test)


Logistic Regression (BoW)
Accuracy: 0.8725
Precision: 0.8685859772816295
Recall: 0.8801349474102005
F1 Score: 0.8743223262690981

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.86      0.87      4961
           1       0.87      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [18]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

print("Naive Bayes (BoW)")
evaluate_model(nb, X_test_bow, y_test)


Naive Bayes (BoW)
Accuracy: 0.8457
Precision: 0.8497398959583834
Recall: 0.8428259575312562
F1 Score: 0.8462688054199462

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.85      0.85      4961
           1       0.85      0.84      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [19]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

print("Naive Bayes (TF-IDF)")
evaluate_model(nb_tfidf, X_test_tfidf, y_test)


Naive Bayes (TF-IDF)
Accuracy: 0.8519
Precision: 0.8511646269245954
Recall: 0.8557253423298273
F1 Score: 0.8534388916378031

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.85      0.85      4961
           1       0.85      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [20]:
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

print("Decision Tree (BoW)")
evaluate_model(dt, X_test_bow, y_test)


Decision Tree (BoW)
Accuracy: 0.7182
Precision: 0.7225005009016229
Recall: 0.7156181782099623
F1 Score: 0.7190428713858424

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.72      0.72      4961
           1       0.72      0.72      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.72      0.72      0.72     10000



In [22]:
dt_tfidf = DecisionTreeClassifier()
dt_tfidf.fit(X_train_tfidf, y_train)

print("Decision Tree (TF-IDF)")
evaluate_model(dt_tfidf, X_test_tfidf, y_test)


Decision Tree (TF-IDF)
Accuracy: 0.7163
Precision: 0.7235987002437043
Recall: 0.7070847390355229
F1 Score: 0.7152464117233764

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.73      0.72      4961
           1       0.72      0.71      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.72      0.72      0.72     10000

